从X生成Q/K/V


In [ ]:
"""
Shape:
X: [B, T, d_model]
Wq: [d_model, d_k]
Wk: [d_model, d_k]
Wv: [d_model, d_v]

Q = XWq -> [B, T, d_k]
K = XWk -> [B, T, d_k]
V = XWv -> [B, T, d_v]

scores = QK^T / sqrt(d_k) -> [B, T, T]
attn_weights = softmax(scores) -> [B, T, T]
output = attn_weights * V -> [B, T, d_v]
"""

In [1]:
import torch
import math

def softmax(x, dim=-1):
    x_max = torch.max(x, dim=dim, keepdim=True).values
    exp_x = torch.exp(x - x_max)
    return exp_x / torch.sum(exp_x, dim=dim, keepdim=True)

In [2]:
def attention_from_x(X, Wq, Wk, Wv, mask=None):
    d_k = Wk.size(-1)
    Q = X @ Wq
    K = X @ Wk
    V = X @ Wv
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))
    attn_weights = softmax(scores)
    output = attn_weights @ V
    return output, attn_weights

In [3]:
B, T, d_model = 2, 4, 16
d_k, d_v = 8, 8

X = torch.randn(B, T, d_model)
Wq = torch.randn(d_model, d_k)
Wk = torch.randn(d_model, d_k)
Wv = torch.randn(d_model, d_v)

mask = torch.tril(torch.ones(T, T)).unsqueeze(0)

output, attn_weights = attention_from_x(X, Wq, Wk, Wv, mask)

print(output.shape)        # 应该是 [2, 4, 8]
print(attn_weights.shape)  # 应该是 [2, 4, 4]
print(attn_weights[0])

torch.Size([2, 4, 8])
torch.Size([2, 4, 4])
tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [9.2833e-01, 7.1666e-02, 0.0000e+00, 0.0000e+00],
        [1.0000e+00, 3.7668e-07, 9.1118e-17, 0.0000e+00],
        [7.3419e-09, 7.3101e-01, 4.7684e-02, 2.2131e-01]])
